# Train FlashSAC for gate racing on Colab

Trains [FlashSAC](https://github.com/Holiday-Robot/FlashSAC) (Kim et al., arXiv 2604.04539: off-policy SAC with large batches, few updates and bounded norms) on the drone-racing task of the PPO runs (`acmpc/race_env.py`, dodgeros dynamics) with `flashsac/flashsac_race.py`, for PPO's budget of 50M env steps. One seed takes ~45 min on a T4.

**VS Code:** open this notebook, **Select Kernel → Colab → New Colab Server → GPU (T4 or better)**, then **Run All**. Training runs as a background process: interrupting the progress cell does not stop it, re-running the training cell resumes unfinished seeds.

Afterwards, download `flashsac_runs.zip`, unzip it into `acmpc/saved/race/` on the Mac, and evaluate or fly the policies there (see the last cell).

In [ ]:
!nvidia-smi -L || echo "No GPU: switch the runtime to a GPU"

## Setup

In [ ]:
%cd /content
!test -d agile-flight-macos || git clone -q https://github.com/krmanik/agile-flight-macos.git
!cd agile-flight-macos && git pull -q
!test -d acmpc_public || git clone -q https://github.com/uzh-rpg/acmpc_public.git
!cd acmpc_public && git -c url.https://github.com/.insteadOf=git@github.com: submodule update -q --init mpc.pytorch
!cd acmpc_public && git checkout -q -- training_modules && git apply ../agile-flight-macos/patches/acmpc_public.patch
!cd acmpc_public/mpc.pytorch && git checkout -q -- mpc && git apply ../../agile-flight-macos/patches/mpc_pytorch.patch
!test -d FlashSAC || git clone -q https://github.com/Holiday-Robot/FlashSAC.git
!cd FlashSAC && git checkout -q 87edc9061150ae9e962dd84e6544e27a1554b3ab
!pip -q install "stable-baselines3>=2.4" gymnasium tensorboard setproctitle
import os
os.environ["ACMPC_PATH"] = "/content/acmpc_public"
os.environ["FLASHSAC_PATH"] = "/content/FlashSAC"
%cd /content/agile-flight-macos

## Settings

Runs are saved to Google Drive when it can be mounted (survives a disconnect), otherwise to `/content/flashsac_runs`. Seeds run one after the other; each resumes from its last checkpoint when started again and is skipped once finished.

In [ ]:
import glob, json

TRACK = "splits"        # acmpc/tracks/: splits (paper), circle
SEEDS = [0, 1, 2]
ENV_STEPS = 50e6        # PPO: 2000 iterations of 25k samples

LOG_DIR = "/content/flashsac_runs"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    LOG_DIR = "/content/drive/MyDrive/flashsac_race_runs"
except Exception as e:
    print("Google Drive not mounted, saving to", LOG_DIR, "-", e)
os.makedirs(LOG_DIR, exist_ok=True)
print("Runs in", LOG_DIR)

def run_name(seed):
    return f"SAC_{TRACK}_dodgeros_s{seed}"

## Train

In [ ]:
import shlex, signal, subprocess, time

# Stop the training started by an earlier run of this cell; the restart
# resumes from the last checkpoint (the replay buffer starts empty)
subprocess.run(["pkill", "-f", "flashsac/flashsac_race.py"])
time.sleep(5)

cmds = [f"python flashsac/flashsac_race.py --track {TRACK} --env_steps {ENV_STEPS:.0f} --seed {s} "
        f"--log_dir {shlex.quote(LOG_DIR)} --run_name {run_name(s)}" for s in SEEDS]
log = open("/content/train_flashsac.log", "a")
# Own session: interrupting a notebook cell must not stop the training
proc = subprocess.Popen(["bash", "-c", " && ".join(cmds)], stdout=log, stderr=subprocess.STDOUT,
                        start_new_session=True, env=dict(os.environ, PYTHONUNBUFFERED="1"))
print("\n".join(cmds))

Progress (re-run any time; stopping this cell does not stop the training):

In [ ]:
from IPython.display import clear_output

while proc.poll() is None:
    clear_output(wait=True)
    for s in SEEDS:
        path = f"{LOG_DIR}/{run_name(s)}/eval.json"
        if os.path.exists(path):
            e = json.load(open(path))[-1]
            print(f"seed {s}: {e['env_steps'] / 1e6:5.1f}M steps  {e['wall_s'] / 60:4.0f} min  "
                  f"success {e['success_rate']:.0%}  gates {e['gates']:.1f}  lap {e['lap_time']:.2f} s")
    print(open("/content/train_flashsac.log").read()[-600:])
    time.sleep(60)
print("exit", proc.returncode); print(open("/content/train_flashsac.log").read()[-1500:])

Optional: TensorBoard (`race/success`, `eval/success_rate`, `train/*`).

In [ ]:
%load_ext tensorboard
%tensorboard --logdir $LOG_DIR

## Evaluate (best checkpoint of each seed)

In [ ]:
runs = [f"{LOG_DIR}/{run_name(s)}" for s in SEEDS if os.path.exists(f"{LOG_DIR}/{run_name(s)}/eval.json")]
RUNS = " ".join(shlex.quote(r + ":best") for r in runs)
!python acmpc/eval_race.py $RUNS --plot /content/race.png
!python acmpc/eval_race.py $RUNS --sweep mass_scale 0.8 1.0 1.2 1.4 --episodes 32
from IPython.display import Image
Image("/content/race.png")

## Package the runs for the Mac (actor checkpoints + logs, without the full agent)

In [ ]:
import shutil, zipfile
zpath = "/content/flashsac_runs.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for run in runs:
        for f in glob.glob(run + "/**", recursive=True):
            rel = os.path.relpath(f, LOG_DIR)
            if os.path.isfile(f) and "/Agent/" not in "/" + rel:
                z.write(f, rel)
print(zpath, os.path.getsize(zpath) // 2**20, "MB")
try:
    from google.colab import files
    files.download(zpath)
except Exception as e:
    print("Automatic download unavailable here:", e)
    print("Get it from Google Drive (copied next to the runs) or the Colab file browser.")
if LOG_DIR.startswith("/content/drive"):
    shutil.copy(zpath, LOG_DIR)

## On the Mac

```bash
unzip flashsac_runs.zip -d acmpc/saved/race/
pixi run setup-flashsac                                   # FlashSAC's network code, to load the actors
pixi run eval-race acmpc/saved/race/SAC_splits_dodgeros_s0:best acmpc/saved/race/MLP_splits_1 --plot race.png
pixi run python flashsac/compare.py "FlashSAC=acmpc/saved/race/SAC_splits_dodgeros_s0" "PPO=acmpc/saved/race/MLP_splits_1"
pixi run python flashsac/benchmark.py "FlashSAC=acmpc/saved/race/SAC_splits_dodgeros_s0:best" "AC-MLP=acmpc/saved/race/MLP_splits_1"
pixi run race-sim --run acmpc/saved/race/SAC_splits_dodgeros_s0:best --render     # ROS sim + Unity
```